# CP2K band unfolding

This viewer loads precomputed unfolding weights retrieved by the CP2K SCF workchain.


In [ ]:
%load_ext aiida
%aiida

import urllib.parse as urlparse

import ipywidgets as ipw
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from aiida import orm


In [ ]:
def _query_pk(default=None):
    try:
        query = urlparse.parse_qs(urlparse.urlparse(jupyter_notebook_url).query)
        if "pk" in query:
            return int(query["pk"][0])
    except Exception:
        pass
    return default

pk = _query_pk()
if pk is None:
    raise ValueError("Open this viewer from a CP2K SCF unfolding result, or pass ?pk=<workchain_pk>.")

workcalc = orm.load_node(pk)
if "unfolding_retrieved" not in workcalc.outputs:
    raise ValueError("This SCF workchain has no unfolding_retrieved output. Submit with Compute band unfolding enabled.")

with workcalc.outputs.unfolding_retrieved.open("unfolding_bands.npz", "rb") as handle:
    loaded = np.load(handle)
    data = {key: loaded[key] for key in loaded.files}

spin_indices = sorted(
    int(key.rsplit("_", 1)[-1])
    for key in data
    if key.startswith("weights_spin_")
)

print(f"SCF workchain PK: {workcalc.pk}")
print("label:", workcalc.label)
print("primitive vectors [Angstrom]:")
print(data["primitive_vectors"])
print("supercell matrix M:")
print(data["supercell_matrix"])
print("det(M):", int(round(abs(np.linalg.det(data["supercell_matrix"])))))
print("folded k-points:", len(data["k_frac_folded"]))
print("k-points on path:", len(data["path_k_indices"]))
print("path:", "-".join(data["path_labels"].astype(str)))


In [ ]:
spin_widget = ipw.Dropdown(options=spin_indices, value=spin_indices[0], description="Spin:")
marker_scale = ipw.FloatSlider(value=220.0, min=10.0, max=800.0, step=10.0, description="Scale:")
show_kmap = ipw.Checkbox(value=True, description="Show k-map")
out = ipw.Output()

def _plot(_=None):
    with out:
        out.clear_output(wait=True)
        spin = spin_widget.value
        weights = data[f"weights_spin_{spin}"]
        energies = data[f"evals_ev_spin_{spin}"] - float(data["ref_energy_ev"])
        path_indices = data["path_k_indices"].astype(int)
        path_x = data["path_x"]
        x_ticks = data["x_ticks"]
        x_labels = data["x_tick_labels"].astype(str)

        if show_kmap.value and int(data["dim"]) == 2:
            k_frac = data["k_frac_folded"] % 1.0
            path_q = data["path_q_equiv"] % 1.0
            fig, ax = plt.subplots(figsize=(4.5, 4.0))
            ax.scatter(k_frac[:, 0], k_frac[:, 1], s=18, alpha=0.45, label="folded k-points")
            if len(path_q):
                ax.scatter(path_q[:, 0], path_q[:, 1], s=42, color="tab:red", label="on plotted path")
            nodes = []
            for label in data["path_labels"].astype(str):
                key = f"hs_point_{label}"
                if key in data:
                    nodes.append(data[key])
                    ax.text(data[key][0], data[key][1], label, ha="center", va="bottom")
            if nodes:
                nodes = np.asarray(nodes)
                ax.plot(nodes[:, 0], nodes[:, 1], color="black", linewidth=1.2)
            ax.set_xlabel("primitive reciprocal fractional k1")
            ax.set_ylabel("primitive reciprocal fractional k2")
            ax.set_xlim(-0.03, 1.03)
            ax.set_ylim(-0.03, 1.03)
            ax.set_aspect("equal", adjustable="box")
            ax.legend(loc="upper right", fontsize=8)
            fig.tight_layout()
            plt.show()

        fig, ax = plt.subplots(figsize=(7, 4))
        for ik, x in zip(path_indices, path_x):
            sizes = marker_scale.value * np.maximum(weights[ik], 0.0)
            mask = sizes > 1.0e-8
            ax.scatter(np.full(np.count_nonzero(mask), x), energies[mask], s=sizes[mask], alpha=0.7, color="black")
        for xt in x_ticks:
            ax.axvline(xt, linewidth=0.8, alpha=0.4)
        ax.axhline(0.0, linestyle="--", linewidth=1)
        ax.set_xticks(x_ticks, x_labels)
        ax.set_xlim(x_ticks[0], x_ticks[-1])
        ax.set_xlabel("primitive-cell k-path")
        ax.set_ylabel("Energy - reference [eV]")
        ax.set_title(f"Unfolded weights, spin {spin}")
        fig.tight_layout()
        plt.show()

for widget in (spin_widget, marker_scale, show_kmap):
    widget.observe(_plot, "value")
display(ipw.HBox([spin_widget, marker_scale, show_kmap]), out)
_plot()
